## Pre-attention mechanism being introduced with LLM's

**RNN's are example which are used to translate from one language to another**

- RNN's used encoder and decoder architecture, unlike LLM's which use decoder only architecture.
- The output from previous steps is fed as input for current step.
- The encoder processes the input text sequencially(one word at a time) and adjusts its hidden state(vector representation/context of the sentence) at each step.
- The output(context) of the encoder is the input for decoder.
- The decoder will build the sentence word by word. It will also adjust it's hidden state which stores the context from encoder and text generated till the current step.
- The decoder could not directly access the earlier hidden states of the encoder but only the final context it has produced. If the input text is long and complex, there will be information loss due to compression of context in each step.

- Later Bahdanau attention mechanism was introduced for RNN's.
- Where decoder part of the network can access all input tokens selectively, that means some inputs are given more importance that the others, which is determined by the attention weights(later).
- For example, for generating the 2nd output token there is more context in the 3rd input token, then it will be given more importance.

## Self Attention

- In self attention we are trying to find the relationship between one part of the input to the other parts of the input.

<img src="images/SelfAttentinOverview.png" width="500">

- In self-attention, our goal is to calculate context vectors z(i) for each element x(i)
in the input sequence. A context vector can be interpreted as an enriched embedding
vector.

- The embedding vector of the second input element, x(2) (which corresponds to the token “journey”), and the corresponding context vector, z(2), shown at the bottom of figure 3.7. This enhanced context vector, z(2), is an embedding that contains information about x(2) and all other input elements, x(1) to x(T).

- Their purpose is to create
enriched representations of each element in an input sequence (like a sentence)
by incorporating information from all other elements in the sequence.

<!-- - Each element in context vector will hold the information about its normal embedding vector and also how it is  -->

<img src="images/SequenceToContextVectors.png" width=600>

In [75]:
import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

<img src="images/AttentionScore.png" width=600>

- Figure 3.8 illustrates how we calculate the intermediate attention scores between the
query token and each input token. We determine these scores by computing the dot
product of the query, x(2), with every other input token.

- Dot product tells us how similar(direction) two vectors are.

In [76]:
query = inputs[1]
atten_score_2 = torch.empty(inputs.shape[0]) # If 10 inputs then we will have 10 attention scores
for i, x in enumerate(inputs):
    atten_score_2[i] = torch.dot(query, x)
print(atten_score_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [77]:
atten_weights_2_tmp = atten_score_2/atten_score_2.sum()
print("Attention weights" ,atten_weights_2_tmp)
print("Sum of weights ", atten_weights_2_tmp.sum())

Attention weights tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum of weights  tensor(1.0000)


- The main goal behind the normalization is to obtain attention
weights that sum up to 1. 

- This normalization is a convention that is useful for interpretation and maintaining training stability in an LLM.

- After doing this we get attention weights.

- In practice it is more advisable to use softmax function to normalize. Which is better at handling extreme values.

In [78]:
atten_weights_2 = torch.softmax(atten_score_2, dim=0)
print("Attention weights: " ,atten_weights_2)
print("Sum: ", atten_weights_2.sum())

Attention weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)


- Context vector is the combined sum of all the products of embedded vectors of the token and the attention weight.

<img src="images/ContextVector.png" width=640>

In [79]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x in enumerate(inputs):
    context_vec_2 += atten_weights_2[i] * x
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


Now we will calculate the attention weights for all input tokens

<img src="images/AttentionMatrix.png" width=600>

In [80]:
atten_scores = torch.zeros((6,6))
for i, x in enumerate(inputs):
    for j, y in enumerate(inputs):
        atten_scores[i][j] = torch.dot(x, y)
print(atten_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


Instead of using for loops as they are slow we use matrix multiplication.

In [81]:
atten_scores = inputs @ inputs.T
print(atten_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [82]:
atten_weights = torch.softmax(atten_scores, dim=-1)
print(atten_weights)
print(atten_weights[0].sum())

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor(1.)


In [83]:
context_vec = atten_weights @ inputs
context_vec

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

## Computing the attention weights step by step

In [84]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

Now we initialize 3 weight matrices which will be optimized during training.

In [85]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [86]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


Weight parameters vs. attention weights
In the weight matrices W, the term “weight” is short for “weight parameters,” the val-
ues of a neural network that are optimized during training. This is not to be confused
with the attention weights. As we already saw, attention weights determine the extent
to which a context vector depends on the different parts of the input (i.e., to what
extent the network focuses on different parts of the input).
In summary, weight parameters are the fundamental, learned coefficients that define
the network’s connections, while attention weights are dynamic, context-specific values.

In [87]:
keys = inputs @ W_key
values = inputs @ W_value

print(keys.shape)
print(values.shape)

torch.Size([6, 2])
torch.Size([6, 2])


Here the query is of context vector 2

<br>
<img src="images/Attention-2.png" width=600>

In [88]:
atten_scores_2 = query_2 @ keys.T
print(atten_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [89]:
d_k = keys.shape[-1]
atten_weights_2 = torch.softmax(atten_scores_2 / d_k ** 0.5 , -1)
print(atten_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


### The rationale behind scaled-dot product attention
The reason for the normalization by the embedding dimension size is to improve the
training performance by avoiding small gradients. For instance, when scaling up the
embedding dimension, which is typically greater than 1,000 for GPT-like LLMs, large
dot products can result in very small gradients during backpropagation due to the
softmax function applied to them. As dot products increase, the softmax function
behaves more like a step function, resulting in gradients nearing zero. These small
gradients can drastically slow down learning or cause training to stagnate.
The scaling by the square root of the embedding dimension is the reason why this
self-attention mechanism is also called scaled-dot product attention.

In [90]:
context_vec_2 = atten_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


- We create a query for each token.

- The query layer is used to know what information does this token need from the sentence. (For it as an example "it" token might be searching for a noun, not in a literal sense but as training goes it will reach a mathamatical conclusion which makes it see "it" as looking for somthing which is similar to a noun). This layer is used to formulate a question.

- In the key layer, if "it" is the query token then noun/subject token like "cat" might match more strongly. Keys can be seen as somthing that describes a token(Mathamatically explaining the "it" and noun relation).

- Value layer contains the actual information to retrive. Values are what that actually gets combined to produce the output representation.(If the model attends to “cat”, the value vector of “cat” contributes to the output).

- These layers are trained and during the training they develop these relationships.

- Each embedding vector is transformed into different vector by multipling them to different matrices.

- These matrices are learned over time during training.

- These learned matrices will transform the vectors into vectors which are better suited for quering, key and values.

- The querying matrix gets it's querying property via weights being changed through backpropogation. The operations which we do in between like finding the scores of tokens and before that finding how similar the query vector is to the key by doing dot product gives it that charactersitic.

```
Suppose we process the token “it”.
-- Step 1 — create query --
q_it

-- Step 2 — compare with all keys -- 
q_it ⋅ k_the
q_it ⋅ k_cat
q_it ⋅ k_sat

-- Step 3 — convert scores to weights -- 
cat → 0.8
sat → 0.1
the → 0.1

-- Step 4 — combine values -- 
output_it =
0.8 * v_cat
+0.1 * v_sat
+0.1 * v_the

So “it” now contains information from “cat.”
```

In [91]:
import torch.nn as nn 
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_values = nn.Parameter(torch.rand(d_in, d_out))
        
    def forward(self, X):
        keys = X @ self.W_key
        queries = X @ self.W_query
        values = X @ self.W_values 

        atten_scores = queries @ keys.T
        atten_weights = torch.softmax(atten_scores/keys.shape[-1] ** 0.5, dim=-1)
        context_vec = atten_weights @ values

        return context_vec

In [92]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


- We use Linear because it is more efficient in the future it will help us in training the weights.

- The initial weights are gotten in a different way compared to the previous version that is why we get a different output vector.

In [93]:
import torch.nn as nn 
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # bias is c in y = m_1 * x_1 + .... + c
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_values = nn.Linear(d_in, d_out, bias=qkv_bias)
        
    def forward(self, X):
        keys = self.W_key(X) # X @ self.W_key
        queries = self.W_query(X)
        values = self.W_values(X) 

        atten_scores = queries @ keys.T
        atten_weights = torch.softmax(atten_scores/keys.shape[-1] ** 0.5, dim=-1)
        context_vec = atten_weights @ values

        return context_vec

In [94]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## Causal attention

- It is a special form of self attention.

- We dont want out LLM to see what tokens are there after the present token which it is processing because it will basically show the answer to the question which we are trying to make the LLM answer(next word prediction).

- We use causal attention/masked attention to hide the upcoming tokens.

<img src="images/CasualAttention1.png" width=600>

- We perform normalization to make the sum of non masked terms in a row to be 1.

<img src="images/CasualMasking.png" width=700>

In [95]:
queries = sa_v2.W_query(inputs) # reusing this from the previous class for convinience
keys = sa_v2.W_key(inputs)
atten_scores = queries @ keys.T
atten_weights = torch.softmax(atten_scores / keys.shape[-1] ** 0.5 , dim=-1)
print(atten_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [96]:
context_length = atten_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length) )
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [97]:
x = torch.tril(atten_weights)
print(x)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<TrilBackward0>)


In [98]:
masked_simple = atten_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [99]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


- It is more efficient to make the elements above the diagonal -inf so we can directly apply softmax and make them 0 instead of finding the sum of the row and then normalizing.

In [100]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1) # this is triu not tril as above this returns a upper matrix
masked = atten_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [101]:
atten_weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=-1)
print(atten_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### Masking additional attention weights with dropout

- This method helps prevent overfitting by ensuring that a model does not become overly reliant on any specific set of hidden layer units.

- Here we are dropping 50% but in production it is usually 10% to 20%.

<img src="images/DropoutMask.png" width=600>

In [102]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # After droping out is done 2(in this case as we choose 0.5 -> 1/2)
                                # it automatically scaled up by 2 to maintain overall balance of 
                                # attention weights ensuring that the average influence of the attention mechanism remains
                                # consistent during both the training and inference phases. 
example = torch.ones(6,6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [103]:
# Complete class
# We want it to be able to handle batches which come from data loader
# For an example we do not use data loader instead we stack 2 inputs on top of eachother

batch = torch.stack((inputs, inputs), dim=0) # 2 inputs each with 6 tokens, each token is a 3d vector
print(batch.shape)

torch.Size([2, 6, 3])


In [104]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

        # register_buffer ensures that the registerd tensor is present on the same device as the LLM(on CPU or GPU)
        # avoiding device mismatch error
    
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        atten_scores = queries @ keys.transpose(1, 2)
        atten_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) # function with _ at the end will do inplace operation
        # Mask can be of size (10, 10) but number 
        # of tokens might be 6(it will create weight 
        # matrices of 6 * 6), so we need to use only 
        # some part of it.
        atten_weights = torch.softmax(atten_scores / keys.shape[-1] ** 0.5, dim=-1)
        atten_weights = self.dropout(atten_weights)

        context_vec = atten_weights @ values
        return context_vec

In [105]:
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0)
context_vec = ca(batch)
print(context_vec.shape)

torch.Size([2, 6, 2])


## Multihead attention

- Attention mechanism is run multiple times in parallel with different learned projections(weight matrices).

<img src="images/MultiHead.png" width=650>

<img src="images/MultiHeadOutput.png" width=500>

- Each head learns a different part of the pattern, like one head will learn the grammer, another head will learn about the subjects in the sentence etc.

In [106]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [116]:
torch.manual_seed(123)
context_length = batch.shape[1] # number of tokens
d_in, d_out = 3, 2
batch.shape

torch.Size([2, 6, 3])

In [117]:
mha = MultiHeadAttentionWrapper(
d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


Exercise 3.2

In [120]:
mha_eg = MultiHeadAttentionWrapper(
d_in, 1, context_length, 0.0, num_heads=2
)
context_vecs_eg = mha_eg(batch)
print(context_vecs_eg)
print("context_vecs.shape:", context_vecs_eg.shape)

tensor([[[0.2847, 0.4336],
         [0.3903, 0.5530],
         [0.4243, 0.5928],
         [0.4009, 0.5332],
         [0.3446, 0.5204],
         [0.3650, 0.4982]],

        [[0.2847, 0.4336],
         [0.3903, 0.5530],
         [0.4243, 0.5928],
         [0.4009, 0.5332],
         [0.3446, 0.5204],
         [0.3650, 0.4982]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


In [125]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
                        "mask",
                        torch.triu(torch.ones(context_length, context_length),
                        diagonal=1))
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(
            b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )
        context_vec = self.out_proj(context_vec)
        return context_vec

- In the above code instead of using different instances of causal attention we use and compute sepeately and adding them at the end, we calculate the entire thing in a single instance.

- We split the output into getting the concatinated vectors as we have seen in the previous version using transpose and reshaping.